# 01 · Data inventory and event register

This notebook catalogues the data for the S&P 500 index-effect study and builds the **event register**: one table of additions and one of deletions, with the flags that define the main samples. `02_post_inclusion.ipynb` runs this notebook and uses the register for the event study and the calendar-time portfolios.

The core sample is 1976–2022, where the membership export is complete. Events in 2023–2025 come from manually collected, provisional files. They are used only as a separately labelled robustness check. The anticipation features and labels at the end of the notebook form an appendix used by notebooks 04 and 05. They are not part of the headline results.

Place the original files in `Data/` at the repository root. Run the notebook from the root or the `notebooks/` directory using the project's Python kernel. Results stay in memory, and nothing is written to disk. Clear cell outputs before sharing the notebook to keep WRDS data local.

In [ ]:
from pathlib import Path

import pandas as pd

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "notebooks" / "01_inventory.ipynb").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the notebook from the AppliedInvestments root or the notebooks directory.")

data_dir = project_root / "Data"
sources = pd.DataFrame([
    ("sp500_snapshot", "Index membership", "S&P 1975 Wharton Research Data Services.xlsx", "Initial index membership with PERMNO"),
    ("sp500_history", "Index membership", "sp500_constituent_changes.csv", "Historical membership spells"),
    ("sp500_additions", "Index membership", "sp500_additions_2023_2025.csv", "Recent additions by effective date"),
    ("sp500_removals", "Index membership", "sp500_removals_2023_2025.csv", "Recent removals with partial coverage"),
    ("crsp_names", "CRSP", "crsp_security_history.csv", "Historical tickers and security attributes"),
    ("crsp_monthly", "CRSP", "crsp_monthly_stock_returns.csv", "Monthly returns and market capitalizations"),
    ("crsp_daily", "CRSP", "crsp_daily_stock_returns_and_quotes.csv", "Daily returns, prices, volume, and bid and ask quotes"),
    ("crsp_delist", "CRSP", "crsp_delisting_events.csv", "Delisting dates, reasons, and returns"),
    ("index_monthly", "Benchmark", "crsp_monthly_index_returns.csv", "S&P 500 and CRSP market series, monthly"),
    ("index_daily", "Benchmark", "crsp_daily_index_returns.csv", "S&P 500 and CRSP market series, daily"),
    ("ccm_link", "Compustat", "crsp_compustat_link_history.csv", "Dated GVKEY–PERMNO link"),
    ("comp_quarterly", "Compustat", "compustat_quarterly_fundamentals.csv", "Quarterly financials"),
    ("comp_annual", "Compustat", "compustat_annual_fundamentals.csv", "Annual financials, including assets, earnings, and buybacks"),
], columns=["dataset", "category", "filename", "usage"]).set_index("dataset")

paths = {dataset: data_dir / filename for dataset, filename in sources["filename"].items()}
legacy_filenames = {
    "crsp_names": "honhxqxgerziffrf.csv",
    "crsp_monthly": "jhvzegjsguw3gw3k.csv",
    "crsp_daily": "wxqvlgh1rjhdvaei.csv",
    "crsp_delist": "zopuwel31lgljgik.csv",
    "index_monthly": "pc8rgx9sahq2tuh6.csv",
    "index_daily": "z8pjwhvfhit5vwei.csv",
    "ccm_link": "kg7lz7q7gsbmphju.csv",
    "comp_quarterly": "pqwyvdhxalbjomaa.csv",
    "comp_annual": "rdmfynls1xcgmjjb.csv",
}
for dataset, filename in legacy_filenames.items():
    legacy_path = data_dir / filename
    if not paths[dataset].is_file() and legacy_path.is_file():
        paths[dataset] = legacy_path

sources["local_filename"] = [path.name for path in paths.values()]
sources["available"] = [path.is_file() for path in paths.values()]
sources["MiB"] = [
    round(path.stat().st_size / 1024**2, 2) if path.is_file() else pd.NA
    for path in paths.values()
]
display(sources[["category", "local_filename", "usage", "available", "MiB"]])
missing_files = sources.loc[~sources["available"], "filename"].tolist()
if missing_files:
    raise FileNotFoundError(f"Missing files in {data_dir}. {', '.join(missing_files)}")

## Initial membership and historical records

The workbook provides the starting composition in early 1975. An `Added` row and a `Removed` row can describe the same membership spell. A spell is identified by `PERMNO, start, end`. The latest end date marks the coverage boundary of the historical export.

In [ ]:
sample_start = pd.Timestamp("1975-01-01")
snapshot = pd.read_excel(paths["sp500_snapshot"], header=1)
for column in ["SP500 Start", "SP500 End"]:
    snapshot[column] = pd.to_datetime(
        snapshot[column].str.replace(".", "", regex=False), format="mixed"
    )

history = pd.read_csv(paths["sp500_history"], parse_dates=["SP500 Start", "SP500 End"])
history = history.rename(columns={
    "PERMNO": "permno", "Ticker": "ticker", "Company": "company",
    "SP500 Start": "start", "SP500 End": "end",
})
spell_key = ["permno", "start", "end"]
historical_spells = history.drop_duplicates(spell_key).sort_values(spell_key)
history_end = historical_spells["end"].max()
snapshot_ids = set(snapshot.loc[
    snapshot["SP500 Start"].le(sample_start) & snapshot["SP500 End"].ge(sample_start), "PERMNO"
])
historical_start_ids = set(historical_spells.loc[
    historical_spells["start"].le(sample_start) & historical_spells["end"].ge(sample_start), "permno"
])
previous_end = historical_spells.groupby("permno")["end"].shift()
display(pd.Series({
    "Securities in the workbook at the start of 1975": len(snapshot_ids),
    "Securities in the historical export at that date": len(historical_start_ids),
    "Workbook securities missing from the historical export at that date": len(snapshot_ids - historical_start_ids),
    "Rows in the historical export": len(history),
    "Distinct membership spells": len(historical_spells),
    "Overlapping consecutive spells for the same security": int(
        previous_end.ge(historical_spells["start"]).sum()
    ),
    "End of historical coverage": history_end.date(),
}, name="check"))

## Match recent additions and removals

A ticker matches a PERMNO when the CRSP ticker validity interval covers the effective date. The public S&P notice seed supplements the supplied additions when a notice identifies an omitted company. For removals, a window of ±10 days identifies potential matches when a ticker stops trading near the effective date. The diagnostic shows how many matches remain ambiguous.

In [ ]:
names = pd.read_csv(
    paths["crsp_names"],
    usecols=["permno", "ticker", "secinfostartdt", "secinfoenddt"],
    parse_dates=["secinfostartdt", "secinfoenddt"],
).dropna(subset=["ticker"])
names["ticker"] = names["ticker"].str.upper().str.strip()

recent_additions = pd.read_csv(
    paths["sp500_additions"], parse_dates=["effective_date", "announcement_date"]
)
recent_removals = pd.read_csv(paths["sp500_removals"], parse_dates=["effective_date"])

notice_key = ["ticker", "effective_date"]
public_notices = pd.read_csv(
    project_root / "references" / "sp500_announcement_seed.csv",
    parse_dates=["effective_date", "announcement_date", "midcap400_exit_date", "narrative_replacement_date"],
)
if public_notices.duplicated(notice_key).any():
    raise ValueError("The S&P notice seed contains a repeated ticker and effective date.")
supplements = public_notices.loc[public_notices["company"].notna(), [
    *notice_key, "announcement_date", "company", "event_type",
]]
missing_locally = ~pd.MultiIndex.from_frame(supplements[notice_key]).isin(
    pd.MultiIndex.from_frame(recent_additions[notice_key])
)
recent_additions = pd.concat([recent_additions, supplements.loc[missing_locally].assign(
    source="sp_dji_notice", migration_from_midcap400="unknown",
)], ignore_index=True)


def ticker_candidates(events, tolerance_days=0):
    keys = events[["ticker", "effective_date"]].rename_axis("event_id").reset_index()
    keys["ticker"] = keys["ticker"].str.upper().str.strip()
    matches = keys.merge(names, on="ticker", how="left")
    tolerance = pd.Timedelta(days=tolerance_days)
    valid = matches["effective_date"].between(
        matches["secinfostartdt"] - tolerance, matches["secinfoenddt"] + tolerance
    )
    return matches.loc[valid, ["event_id", "permno"]].drop_duplicates()


def candidate_counts(events, matches):
    return matches.groupby("event_id")["permno"].size().reindex(events.index, fill_value=0)


add_matches = ticker_candidates(recent_additions)
add_counts = candidate_counts(recent_additions, add_matches)
unique_add_matches = add_matches.loc[add_matches["event_id"].isin(add_counts[add_counts.eq(1)].index)]
recent_additions["permno"] = recent_additions.index.to_series().map(
    unique_add_matches.set_index("event_id")["permno"]
).astype("Int64")

removal_counts = candidate_counts(recent_removals, ticker_candidates(recent_removals))
removal_counts_with_tolerance = candidate_counts(
    recent_removals, ticker_candidates(recent_removals, tolerance_days=10)
)
match_counts = pd.DataFrame({
    "Additions, exact date": add_counts.value_counts(),
    "Removals, exact date": removal_counts.value_counts(),
    "Removals, ±10 days": removal_counts_with_tolerance.value_counts(),
}).fillna(0).astype(int).sort_index()
display(match_counts.rename_axis("Candidate PERMNOs"))
display(pd.DataFrame({
    "Additions": recent_additions.groupby(recent_additions["effective_date"].dt.year).size(),
    "Removals": recent_removals.groupby(recent_removals["effective_date"].dt.year).size(),
}))

## Additions from the membership spells and the recent files

`events` collects the additions: membership spells that start in 1975 or later, plus the provisional 2023–2025 additions file supplemented by the S&P notice seed. It is the input to the event register below. The `announcement_date` column here comes from the provisional additions file and is **not verified**. The event register keeps only the dates documented in the S&P notice seed. The migration status remains `unknown` when the source provides no classification.

The twelve-month column measures calendar coverage after the inclusion month only. Return coverage and the event filters are applied in the event register and in notebook 02.

In [ ]:
historical_additions = historical_spells.loc[
    historical_spells["start"].between(sample_start, history_end),
    ["permno", "ticker", "company", "start"],
].rename(columns={"start": "effective_date"}).assign(
    source="historical_membership_export",
    event_type="unknown",
    migration_from_midcap400="unknown",
    announcement_date=pd.NaT,
)
event_columns = [
    "permno", "ticker", "company", "effective_date", "announcement_date",
    "event_type", "migration_from_midcap400", "source",
]
events = pd.concat(
    [historical_additions[event_columns], recent_additions[event_columns]],
    ignore_index=True,
).sort_values(["effective_date", "permno"], ignore_index=True)
events["permno"] = events["permno"].astype("Int64")
resolved = events.loc[events["permno"].notna()]
if resolved.duplicated(["permno", "effective_date"]).any():
    raise ValueError("Multiple additions share the same PERMNO and effective date.")

calendar_dates = {}
for dataset, date_column in [("index_daily", "dlycaldt"), ("index_monthly", "mthcaldt")]:
    calendar_dates[dataset] = pd.read_csv(
        paths[dataset], usecols=[date_column], parse_dates=[date_column]
    )[date_column]
display(pd.DataFrame({
    dataset: {"first date": dates.min().date(), "last date": dates.max().date()}
    for dataset, dates in calendar_dates.items()
}).T)

last_month = calendar_dates["index_monthly"].max().to_period("M")
events["calendar_has_12_months"] = events["effective_date"].dt.to_period("M") + 12 <= last_month
events["period"] = pd.cut(
    events["effective_date"].dt.year,
    bins=[1974, 1999, 2009, 2017, 2025],
    labels=["1975–1999", "2000–2009", "2010–2017", "2018–2025"],
)
display(events.groupby("period", observed=True).agg(
    additions=("effective_date", "size"),
    with_permno=("permno", "count"),
    with_announcement_date=("announcement_date", "count"),
    reported_migrations=("migration_from_midcap400", lambda values: values.eq("yes").sum()),
    unknown_migration_status=("migration_from_midcap400", lambda values: values.eq("unknown").sum()),
    full_12_month_calendar=("calendar_has_12_months", "sum"),
))

## Optional audit of the monthly export

This cell reads about 371 MB. Set `audit_monthly` to `True` to run it. The audit compares market fields while omitting ticker and other redundant identifiers. After dropping identical records on those fields, it raises an error if a security and month still have conflicting observations.

In [ ]:
audit_monthly = False
if audit_monthly:
    monthly_key = ["PERMNO", "MthCalDt"]
    monthly_columns = monthly_key + ["MthPrc", "MthDelFlg", "MthCap", "MthRet", "MthRetx", "MthVol"]
    monthly_raw = pd.read_csv(
        paths["crsp_monthly"], usecols=monthly_columns, parse_dates=["MthCalDt"]
    )
    monthly_market = monthly_raw.drop_duplicates()
    if monthly_market.duplicated(monthly_key).any():
        raise ValueError("Conflicting market observations for the same security and month.")
    display(pd.Series({
        "Export rows": len(monthly_raw),
        "Duplicate rows on market fields": len(monthly_raw) - len(monthly_market),
        "Missing monthly returns after removing duplicates": int(monthly_market["MthRet"].isna().sum()),
        "First date": monthly_market["MthCalDt"].min().date(),
        "Last date": monthly_market["MthCalDt"].max().date(),
    }, name="check"))
    del monthly_raw

In [ ]:
from csv import reader
from io import TextIOWrapper
from zipfile import ZipFile


def read_monthly_factors(archive_path, columns):
    with ZipFile(archive_path) as archive:
        filename, = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        with archive.open(filename) as file:
            rows = ([value.strip() for value in row] for row in reader(TextIOWrapper(file, encoding="utf-8-sig")))
            header = next(row for row in rows if set(columns).issubset(row))
            monthly_rows = [row for row in rows if row and len(row[0]) == 6 and row[0].isdigit()]

    factors = pd.DataFrame(monthly_rows, columns=["yyyymm", *header[1:]])
    factors["month"] = pd.to_datetime(factors.pop("yyyymm"), format="%Y%m").dt.to_period("M")
    values = factors.set_index("month")[columns].astype(float)
    return values.mask(values.isin([-99.99, -999])).div(100)

## Data limits

- **Membership.** The historical export is complete through December 30, 2022, and defines the core sample (1976–2022). It omits some spells of securities that were members in 1975. The 1975 workbook, a later vintage of the same spells, fills them in (see the event register). Additions and removals for 2023–2025 come from manually collected files and are provisional. The removals file ends in August 2025 and omits at least one 2024 removal. The [December 5, 2025 announcement](https://www.spglobal.com/spdji/en/documents/indexnews/announcements/20251205-1480992/1480992_december2025rebalance.pdf) confirms three further removals effective December 22, 2025.
- **Date precision.** Before about 1990, the export records membership changes on a weekly grid (almost always a Wednesday). Some exits are recorded up to several weeks after the security's CRSP delisting. Effective dates in 1976–1989 are therefore imprecise by up to about a week.
- **Announcement dates.** Verified announcement dates exist only for the 19 additions in the S&P notice seed, all from 2025. No core-sample event has one, so event windows are centred on the index trading date.
- **Event types.** The export does not classify events. Spin-offs, merger successors, and delisting-related deletions are identified with CRSP proxies (see the event register). S&P MidCap 400 membership is not available, so migrations cannot be identified.
- **Fundamentals.** Compustat is used only in the anticipation appendix. It is the current, restated vintage, not point-in-time data.
- **Delisting returns.** CRSP flags missing or partial delisting returns. Notebook 02 reports a sensitivity check that replaces them with an assumed return. [WRDS documentation on delisting returns in CIZ](https://wrds-www.wharton.upenn.edu/pages/wrds-research/macros/run-an-event-study-ciz-format-macro/).

## Monthly panel, membership, and the spin-off proxy

The monthly CRSP panel records S&P 500 membership at each month-end (workbook spells capped at the export boundary plus the historical spells, both endpoints inclusive) and eligibility (U.S. ordinary common equity on NYSE, NYSE American, or Nasdaq). The cell also adds `likely_spin_or_merger`, a proxy for spin-offs and merger successors. It flags an addition whose security has no CRSP row in the three calendar months before the effective month (see Appendix A.3 for its limits).

In [ ]:
import numpy as np

panel_columns = {
    "PERMNO": "permno", "MthCalDt": "mthcaldt", "MthCap": "mthcap", "MthPrc": "mthprc",
    "MthRet": "mthret", "MthVol": "mthvol", "MthDelFlg": "mthdelflg",
}
monthly_panel = pd.read_csv(
    paths["crsp_monthly"], usecols=list(panel_columns), parse_dates=["MthCalDt"]
).rename(columns=panel_columns).drop_duplicates()
if monthly_panel.duplicated(["permno", "mthcaldt"]).any():
    raise ValueError("Conflicting monthly observations for the same security and month.")
monthly_panel = monthly_panel.sort_values(["permno", "mthcaldt"], ignore_index=True)
panel_rows = monthly_panel[["permno", "mthcaldt"]].reset_index()

# S&P 500 membership: workbook spells capped at the export boundary plus historical spells (inclusive).
panel_spells = pd.concat([
    historical_spells[["permno", "start", "end"]],
    snapshot.rename(columns={"PERMNO": "permno", "SP500 Start": "start", "SP500 End": "end"})
    [["permno", "start", "end"]].assign(end=lambda s: s["end"].clip(upper=history_end)),
], ignore_index=True).drop_duplicates()
spell_matches = panel_rows.merge(panel_spells, on="permno")
member_rows = spell_matches.loc[
    spell_matches["mthcaldt"].between(spell_matches["start"], spell_matches["end"]), "index"
].unique()
monthly_panel["in_sp500"] = monthly_panel.index.isin(member_rows)

# Eligibility as in notebook 03: U.S. ordinary common equity on NYSE, NYSE American, or Nasdaq.
security_terms = pd.read_csv(
    paths["crsp_names"],
    usecols=["permno", "secinfostartdt", "secinfoenddt", "usincflg",
             "securitytype", "securitysubtype", "sharetype", "primaryexch"],
    parse_dates=["secinfostartdt", "secinfoenddt"],
)
eligible_terms = security_terms.loc[
    security_terms["usincflg"].eq("Y") & security_terms["securitytype"].eq("EQTY")
    & security_terms["securitysubtype"].eq("COM") & security_terms["sharetype"].eq("NS")
    & security_terms["primaryexch"].isin(["N", "A", "Q"]),
    ["permno", "secinfostartdt", "secinfoenddt"],
].sort_values("secinfostartdt")
dated_terms = pd.merge_asof(
    panel_rows.sort_values("mthcaldt"), eligible_terms,
    left_on="mthcaldt", right_on="secinfostartdt", by="permno", direction="backward",
)
eligible_rows = dated_terms.loc[dated_terms["mthcaldt"].le(dated_terms["secinfoenddt"]), "index"]
monthly_panel["eligible"] = (
    monthly_panel.index.isin(eligible_rows)
    & monthly_panel["mthcap"].gt(0) & monthly_panel["mthdelflg"].eq("N")
)

# Contamination proxy: no CRSP row in the three calendar months before the effective month.
# Uses only rows dated before the effective month. Events too early to assess stay unflagged.
crsp_months = monthly_panel[["permno"]].assign(
    panel_month=monthly_panel["mthcaldt"].dt.year * 12 + monthly_panel["mthcaldt"].dt.month
)
event_month = events["effective_date"].dt.year * 12 + events["effective_date"].dt.month
event_rows = events.loc[events["permno"].notna(), ["permno"]].astype("int64").assign(
    event_month=event_month
).reset_index().merge(crsp_months, on="permno")
has_prior_row = event_rows.loc[
    (event_rows["event_month"] - event_rows["panel_month"]).between(1, 3), "index"
].unique()
events["contamination_assessable"] = event_month - 3 >= crsp_months["panel_month"].min()
events["likely_spin_or_merger"] = (
    events["permno"].notna() & events["contamination_assessable"] & ~events.index.isin(has_prior_row)
)
display(events.groupby(events["effective_date"].dt.year // 10 * 10).agg(
    additions=("effective_date", "size"),
    not_assessable=("contamination_assessable", lambda values: (~values).sum()),
    flagged_excluded=("likely_spin_or_merger", "sum"),
).rename_axis("decade"))

del panel_rows, spell_matches, dated_terms, security_terms

## Event register: additions and deletions

This register is the sample for the index-effect study. It has one table of **additions** and one of **deletions**, both built from the membership spells.

| Field | Additions | Deletions |
| --- | --- | --- |
| `effective_date` | Spell start, the first session in the index | Spell end, the last session in the index |
| `index_trade_date` | Session before `effective_date`, whose close index funds use to buy | `effective_date`, whose close index funds use to sell |

The historical export confirms the convention. For 96% of deletion end dates, a replacement addition starts on the next session. The 2023–2025 removals CSV reports S&P's effective date, the first session *out* of the index. That date is converted to the previous session, so `effective_date` means the same thing in both sources. The original date is kept as `sp_effective_date`.

**Samples.** `core` is 1976–2022, where the membership export is complete. `robustness_2023_2025` uses the manual CSVs and the S&P notice seed. Those are provisional: removals end in August 2025, and tickers are matched to PERMNOs by date. Events in 1975 are labelled `pre_core_1975` and are not used.

**Flags.** Every flag uses only information available at the event date. No flag depends on how long the stock later stays in the index.

- `flag_no_prior_crsp_3m` (additions): the security has no CRSP monthly row in the three calendar months before the effective month. This is a proxy for spin-offs and merger successors. It can also catch direct listings and new PERMNOs from reorganisations. Five additions in early 1975 cannot be assessed.
- `flag_delisting_related` (deletions): the CRSP delisting date falls before the deletion, or at most 21 sessions after it. These removals follow an acquisition, bankruptcy, or merger and are not pure index deletions. A deal that has closed within a month was usually public at the deletion date. The delisting date is the proxy for that information. The table below reports counts for windows of 5 and 63 sessions.
- `flag_no_prior_crsp_3m` (deletions): the same CRSP-history proxy applied to deletions. It identifies spin-offs that S&P adds on the distribution date and removes a few days later (19 main-sample deletions, all in 2015–2022). The flag was added after the data were inspected. It defines a robustness check in notebook 02 and does **not** change the main sample.
- `main_sample`: core events with a PERMNO. For deletions, they must also not be delisting related, so the stock keeps trading. For additions, `flag_no_prior_crsp_3m` must be false.

**Announcement dates.** Only dates from the S&P notice seed (`references/sp500_announcement_seed.csv`) are kept. Those 19 dates are all 2025 additions. The Wikipedia dates in the additions CSV are not verified and are dropped. **No core-sample event has a verified announcement date.**

In [ ]:
register_columns = [
    "permno", "ticker", "company", "event_type", "effective_date", "index_trade_date",
    "source", "sample",
]
trading_calendar = pd.DatetimeIndex(calendar_dates["index_daily"]).sort_values()
session_of = pd.Series(np.arange(len(trading_calendar)), index=trading_calendar)
core_start, core_end = pd.Timestamp("1976-01-01"), history_end
delisting_window_sessions = 21


def sample_label(dates):
    return pd.Series(np.select(
        [dates.lt(core_start), dates.le(core_end)],
        ["pre_core_1975", "core"], default="robustness_2023_2025",
    ), index=dates.index)


def shift_sessions(dates, offset):
    positions = trading_calendar.searchsorted(dates) + offset
    if (positions < 0).any() or (positions >= len(trading_calendar)).any():
        raise ValueError("A shifted event date falls outside the daily trading calendar.")
    return pd.Series(trading_calendar[positions], index=dates.index)


# Additions: historical spell starts from 1975 plus the 2023-2025 CSV and the notice seed (from `events`).
additions_register = events.loc[events["effective_date"].ge(sample_start)].copy()
if not additions_register["effective_date"].isin(trading_calendar).all():
    raise ValueError("An addition date is not a trading session.")
additions_register["event_type"] = "addition"
additions_register["index_trade_date"] = shift_sessions(additions_register["effective_date"], -1)
additions_register["sample"] = sample_label(additions_register["effective_date"])
additions_register["flag_no_prior_crsp_3m"] = additions_register["likely_spin_or_merger"]
additions_register["flag_not_assessable"] = ~additions_register["contamination_assessable"]
# Only the S&P notice seed counts as a verified announcement date.
additions_register = additions_register.drop(columns="announcement_date").merge(
    public_notices[[*notice_key, "announcement_date", "effective_timing", "source_url"]].rename(columns={
        "announcement_date": "verified_announcement_date", "source_url": "announcement_source_url",
    }),
    on=notice_key, how="left", validate="many_to_one",
)
reported_types = recent_additions.set_index(notice_key)["event_type"].rename("reported_event_type_provisional")
additions_register = additions_register.join(reported_types, on=notice_key)
additions_register["main_sample"] = (
    additions_register["sample"].eq("core") & additions_register["permno"].notna()
    & ~additions_register["flag_no_prior_crsp_3m"]
)

# Deletions, historical: spells ending before the export boundary. Spells ending on it are censored.
# The 1975 workbook is a later vintage of the spells of 1975 members. It adds spells the export omits.
workbook_spells = snapshot.rename(columns={
    "PERMNO": "permno", "Ticker": "ticker", "Company Name": "company", "SP500 Start": "start", "SP500 End": "end",
})[["permno", "ticker", "company", "start", "end"]]
workbook_only = workbook_spells.merge(
    historical_spells[["permno", "start"]], on=["permno", "start"], how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")
historical_deletions = pd.concat([
    historical_spells.assign(source="historical_membership_export"),
    workbook_only.assign(source="sp500_1975_workbook"),
], ignore_index=True)
historical_deletions = historical_deletions.loc[
    historical_deletions["end"].ge(sample_start) & historical_deletions["end"].lt(history_end),
    ["permno", "ticker", "company", "end", "source"],
].rename(columns={"end": "effective_date"})
historical_deletions["sp_effective_date"] = shift_sessions(historical_deletions["effective_date"], 1)

# Deletions, 2023-2025: match the ticker to a PERMNO valid within 10 days of S&P's effective date.
# If several PERMNOs match (share classes), keep the one that was a member at the end of 2022
# or was added in 2023-2025. Both facts are known at the removal date.
recent_deletions = recent_removals.rename(columns={"effective_date": "sp_effective_date"}).copy()
recent_deletions["ticker"] = recent_deletions["ticker"].str.upper().str.strip()
removal_matches = ticker_candidates(
    recent_removals.assign(ticker=recent_deletions["ticker"]), tolerance_days=10
)
members_at_history_end = pd.concat([historical_spells, workbook_spells]).loc[
    lambda spells: spells["start"].le(history_end) & spells["end"].ge(history_end), "permno"
]
known_members = set(members_at_history_end) | set(recent_additions["permno"].dropna().astype(int))
removal_matches = removal_matches.loc[
    removal_matches.groupby("event_id")["permno"].transform("size").eq(1)
    | removal_matches["permno"].isin(known_members)
]
if removal_matches["event_id"].duplicated().any():
    raise ValueError("A 2023-2025 removal still matches several PERMNOs.")
recent_deletions["permno"] = removal_matches.set_index("event_id")["permno"]
recent_deletions["effective_date"] = shift_sessions(recent_deletions["sp_effective_date"], -1)
recent_deletions["source"] = "manual_removals_csv_2023_2025"
recent_deletions["removal_not_member_at_2022_end"] = (
    recent_deletions["permno"].notna() & ~recent_deletions["permno"].isin(known_members)
)

# The workbook also records 2023-2024 exits of 1975 members (its last date, 2024-12-31, is censored).
# Add exits the removals CSV omits. Their end dates are already the last session in the index.
workbook_recent_exits = workbook_spells.loc[
    workbook_spells["end"].gt(history_end) & workbook_spells["end"].lt("2024-12-31"),
    ["permno", "ticker", "company", "end"],
].rename(columns={"end": "effective_date"})
exit_in_csv = workbook_recent_exits.merge(
    recent_deletions.dropna(subset=["permno"]).astype({"permno": "int64"})[["permno", "effective_date"]],
    on=["permno", "effective_date"], how="left", indicator=True,
)["_merge"].eq("both").to_numpy()
workbook_added_exits = workbook_recent_exits.loc[~exit_in_csv].assign(source="sp500_1975_workbook")
workbook_added_exits["sp_effective_date"] = shift_sessions(workbook_added_exits["effective_date"], 1)

deletions_register = pd.concat(
    [historical_deletions, recent_deletions, workbook_added_exits], ignore_index=True
)
deletions_register["permno"] = deletions_register["permno"].astype("Int64")
deletions_register["event_type"] = "deletion"
deletions_register["index_trade_date"] = deletions_register["effective_date"]
deletions_register["sample"] = sample_label(deletions_register["effective_date"])

delisting_info = pd.read_csv(
    paths["crsp_delist"],
    usecols=["permno", "delistingdt", "delactiontype", "delreasontype", "delret"],
    parse_dates=["delistingdt"],
)
if delisting_info["permno"].duplicated().any():
    raise ValueError("The delisting file has more than one row per PERMNO.")
deletions_register = deletions_register.merge(
    delisting_info.rename(columns={
        "delistingdt": "delisting_date", "delactiontype": "delisting_action",
        "delreasontype": "delisting_reason", "delret": "delisting_return",
    }).astype({"permno": "Int64"}),
    on="permno", how="left", validate="many_to_one",
)
has_delisting = deletions_register["delisting_date"].notna()
deletions_register["sessions_to_delisting"] = pd.Series(
    trading_calendar.searchsorted(deletions_register.loc[has_delisting, "delisting_date"])
    - deletions_register.loc[has_delisting, "effective_date"].map(session_of).to_numpy(),
    index=deletions_register.index[has_delisting],
).astype("Int64")


def delisting_related(window):
    return deletions_register["sessions_to_delisting"].le(window).fillna(False).astype(bool)


deletions_register["flag_delisting_related"] = delisting_related(delisting_window_sessions)
deletions_register["main_sample"] = (
    deletions_register["sample"].eq("core") & deletions_register["permno"].notna()
    & ~deletions_register["flag_delisting_related"]
)
# Same proxy as for additions: no CRSP row in the three calendar months before the effective month.
# It identifies spin-offs that S&P adds on the distribution date and removes shortly afterwards.
# It was added after inspecting the data, so it defines a robustness check and not the main sample.
deletion_month = deletions_register["effective_date"].dt.year * 12 + deletions_register["effective_date"].dt.month
deletion_rows = deletions_register.loc[deletions_register["permno"].notna(), ["permno"]].astype("int64").assign(
    event_month=deletion_month
).reset_index().merge(crsp_months, on="permno")
has_prior_deletion_row = deletion_rows.loc[
    (deletion_rows["event_month"] - deletion_rows["panel_month"]).between(1, 3), "index"
].unique()
deletions_register["flag_no_prior_crsp_3m"] = (
    deletions_register["permno"].notna() & ~deletions_register.index.isin(has_prior_deletion_row)
)

deletions_register = deletions_register[[
    *register_columns, "sp_effective_date", "delisting_date", "delisting_action",
    "delisting_reason", "delisting_return", "sessions_to_delisting", "flag_delisting_related",
    "flag_no_prior_crsp_3m", "removal_not_member_at_2022_end", "main_sample",
]].sort_values(["effective_date", "permno"], ignore_index=True)
additions_register = additions_register[[
    *register_columns, "flag_no_prior_crsp_3m", "flag_not_assessable",
    "verified_announcement_date", "effective_timing", "announcement_source_url",
    "reported_event_type_provisional", "main_sample",
]].sort_values(["effective_date", "permno"], ignore_index=True)

for register_name, register_table in [("additions", additions_register), ("deletions", deletions_register)]:
    if register_table.dropna(subset=["permno"]).duplicated(["permno", "effective_date"]).any():
        raise ValueError(f"The {register_name} register has a repeated PERMNO and effective date.")


def decade_label(dates, samples):
    decade = (dates.dt.year // 10 * 10).astype(str) + "s"
    return decade.where(samples.eq("core"), samples)


addition_counts = additions_register.groupby(
    decade_label(additions_register["effective_date"], additions_register["sample"]).rename("period")
).agg(
    additions=("effective_date", "size"),
    no_permno=("permno", lambda values: values.isna().sum()),
    no_prior_crsp_3m=("flag_no_prior_crsp_3m", "sum"),
    not_assessable=("flag_not_assessable", "sum"),
    verified_announcement=("verified_announcement_date", "count"),
    main_sample=("main_sample", "sum"),
)
deletion_counts = deletions_register.groupby(
    decade_label(deletions_register["effective_date"], deletions_register["sample"]).rename("period")
).agg(
    deletions=("effective_date", "size"),
    no_permno=("permno", lambda values: values.isna().sum()),
    delisted_before=("sessions_to_delisting", lambda values: values.lt(0).sum()),
    delisted_0_to_21=("sessions_to_delisting", lambda values: values.between(0, 21).sum()),
    delisting_related=("flag_delisting_related", "sum"),
    main_sample_keeps_trading=("main_sample", "sum"),
    main_sample_no_prior_crsp_3m=("flag_no_prior_crsp_3m", lambda values: (
        values & deletions_register.loc[values.index, "main_sample"]
    ).sum()),
)
display(addition_counts)
display(deletion_counts)

# Sensitivity of the delisting flag to the window, core sample, by CRSP delisting action.
core_deletions = deletions_register["sample"].eq("core")
core_actions = deletions_register.loc[core_deletions, "delisting_action"].fillna("none")
display(pd.DataFrame({
    f"flagged with window <= {window} sessions": delisting_related(window)[core_deletions].groupby(core_actions).sum()
    for window in (5, 21, 63)
}).T.assign(total=lambda frame: frame.sum(axis=1)))
display(pd.Series({
    "Unverified reported announcement dates dropped": int(recent_additions["announcement_date"].notna().sum()),
    "2023-2025 removals matched to a member PERMNO": int(
        (recent_deletions["permno"].notna() & ~recent_deletions["removal_not_member_at_2022_end"]).sum()
    ),
    "2023-2025 removals without a matched member PERMNO": int(
        (recent_deletions["permno"].isna() | recent_deletions["removal_not_member_at_2022_end"]).sum()
    ),
    "Last 2023-2025 removal in the CSV": recent_removals["effective_date"].max().date(),
    # Workbook end dates should equal the converted CSV dates. Exits missing from the CSV are added.
    "Workbook exits in 2023-2024": len(workbook_recent_exits),
    "  matching a converted CSV removal (PERMNO and date)": int(exit_in_csv.sum()),
    "  missing from the CSV, added from the workbook": len(workbook_added_exits),
}, name="check"))

## Appendix A · Anticipation inputs (not part of the headline results)

The cells below build the inputs for the optional anticipation appendix in notebooks 04 and 05: monthly labels for an addition that becomes effective within three or six months, and a standardised feature table of eligible non-members. They do not feed the index-effect study in notebook 02. Because no core-sample announcement date is verified, the labels are based on effective dates and cannot separate anticipation from announcement.

### Appendix A.1 · Labels

In [ ]:
panel_rows = monthly_panel[["permno", "mthcaldt"]].reset_index()

# Labels: an addition becomes effective within the next 3 or 6 months (missing if the horizon is incomplete).
label_targets = events.loc[
    events["permno"].notna() & events["effective_date"].le(history_end) & ~events["likely_spin_or_merger"],
    ["permno", "effective_date"]
].astype({"permno": "int64"}).sort_values("effective_date")
next_effective = pd.merge_asof(
    panel_rows.sort_values("mthcaldt"), label_targets,
    left_on="mthcaldt", right_on="effective_date", by="permno",
    direction="forward", allow_exact_matches=False,
).set_index("index")["effective_date"].reindex(monthly_panel.index)
panel_month = monthly_panel["mthcaldt"].dt.to_period("M")
for horizon in (3, 6):
    horizon_month = panel_month + horizon
    monthly_panel[f"y{horizon}"] = (
        next_effective.dt.to_period("M").le(horizon_month)
        .where(horizon_month.le(history_end.to_period("M"))).astype("Int8")
    )
del panel_rows

### Appendix A.2 · Anticipation feature table

Each row is an eligible non-member at month-end `mthcaldt` (January 1975 through the membership boundary at December 2022). Features use only data observed at or before that month-end. Return windows require consecutive months with complete returns. Otherwise the feature is missing.

| Feature | Definition |
| --- | --- |
| `log_mktcap` | Log of CRSP security market capitalization at t |
| `mktcap_pct_rank` | Percentile rank of market cap among non-members that month (1 = largest) |
| `dollar_volume_to_mktcap` | Monthly dollar volume (`mthvol × abs(mthprc)`) divided by market cap, a turnover proxy |
| `log_dollar_volume` | Log of monthly dollar volume |
| `mom_12_1` | Compounded return over months t−11 to t−1, skipping month t |
| `ret_1m` | Return in month t |
| `ret_3m` | Compounded return over months t−2 to t |
| `vol_12m` | Standard deviation of monthly returns over months t−11 to t |
| `log_price` | Log of the absolute month-end price |

Each feature except `mktcap_pct_rank` is winsorized at the 1st and 99th percentiles within its month. The percentile rank is already uniform, so it is not winsorized. All features are then standardized to mean 0 and standard deviation 1 within each month.

`y3` and `y6` are labels, not features. They equal 1 when the security's addition becomes effective within 3 or 6 months after t. The labels are right-censored: in the last 3 months of the panel for `y3`, and the last 6 months for `y6`, the horizon extends past the end of the membership data, so the label is missing. **Drop those rows before fitting a model.** Do not treat them as zeros.

In [ ]:
panel = monthly_panel
security = panel["permno"]
month_number = panel["mthcaldt"].dt.year * 12 + panel["mthcaldt"].dt.month
log_return = np.log1p(panel["mthret"])


def months_since(lag):
    return month_number - month_number.groupby(security).shift(lag)


def trailing(values, window, statistic):
    rolled = getattr(values.groupby(security).rolling(window, min_periods=window), statistic)()
    return rolled.droplevel(0).where(months_since(window - 1).eq(window - 1))


price = panel["mthprc"].abs()
dollar_volume = panel["mthvol"] * price
raw_features = pd.DataFrame({
    "permno": security,
    "mthcaldt": panel["mthcaldt"],
    "log_mktcap": np.log(panel["mthcap"].where(panel["mthcap"].gt(0))),
    "dollar_volume_to_mktcap": dollar_volume / (panel["mthcap"] * 1000),  # MthCap is in $ thousands
    "log_dollar_volume": np.log(dollar_volume.where(dollar_volume.gt(0))),
    "mom_12_1": np.expm1(
        trailing(log_return, 11, "sum").groupby(security).shift(1).where(months_since(1).eq(1))
    ),
    "ret_1m": panel["mthret"],
    "ret_3m": np.expm1(trailing(log_return, 3, "sum")),
    "vol_12m": trailing(panel["mthret"], 12, "std"),
    "log_price": np.log(price.where(price.gt(0))),
    "y3": panel["y3"],
    "y6": panel["y6"],
})

features = raw_features.loc[
    ~panel["in_sp500"] & panel["eligible"] & panel["mthcaldt"].between(sample_start, history_end)
].reset_index(drop=True)
features.insert(3, "mktcap_pct_rank", features.groupby("mthcaldt")["log_mktcap"].rank(pct=True))
feature_columns = [
    "log_mktcap", "mktcap_pct_rank", "dollar_volume_to_mktcap", "log_dollar_volume",
    "mom_12_1", "ret_1m", "ret_3m", "vol_12m", "log_price",
]
winsorized_columns = [column for column in feature_columns if column != "mktcap_pct_rank"]

# Unit check for MthVol before standardizing: median monthly dollar volume / market cap.
# If MthVol is in shares, typical monthly turnover is a few percent (roughly 0.01-0.2).
# If it is in hundreds of shares, the median is about 100 times smaller.
turnover_median = features.groupby(features["mthcaldt"].dt.year // 10 * 10)[
    "dollar_volume_to_mktcap"
].median()
print(f"Median dollar volume / market cap, all months: {features['dollar_volume_to_mktcap'].median():.5f}")
print("By decade:")
print(turnover_median.rename_axis("decade").to_string())

by_month = features.groupby("mthcaldt")[winsorized_columns]
winsorized = features[feature_columns].copy()
winsorized[winsorized_columns] = features[winsorized_columns].clip(
    lower=by_month.transform(lambda column: column.quantile(0.01)),
    upper=by_month.transform(lambda column: column.quantile(0.99)),
)
winsorized_by_month = winsorized.groupby(features["mthcaldt"])
features[feature_columns] = (
    (winsorized - winsorized_by_month.transform("mean")) / winsorized_by_month.transform("std")
)
del raw_features, winsorized

print(f"\nRows: {len(features):,}")
print("\nMissing values per column:")
print(features.isna().sum().to_string())
print("\nSummary statistics:")
print(features[feature_columns + ["y3", "y6"]].describe().T.to_string())

### Appendix A.3 · Note on the spin-off flag

`likely_spin_or_merger` (cell "monthly panel") flags an addition when its security has no CRSP row in the three calendar months before the effective month. It uses only rows dated before the effective month and no information about how long the stock stayed in the index. Flagged additions are excluded from the anticipation labels, from the main event-study sample, and from the calendar-time portfolios. They remain index members, so they also stay out of the anticipation candidate universe.

The flag is a proxy and does not verify the event type. It can also flag direct IPO additions, moves from other markets, and new PERMNOs from reorganisations. It misses events with CRSP history, such as Bemis's three-day spell in 2019, IPOs less than three months before the event, and acquirers, because the project has no M&A data. Five additions from February and March 1975 cannot be assessed and fall outside the core sample.